In [ ]:
import os
import json
import re
import pandas as pd
from normalizer import normalize
from sklearn.model_selection import train_test_split
from datasets import load_dataset

# Text cleaning function
def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return normalize(text)

print("Loading BD-SHS and BIDWESH for exclusion matching...")
bdshs_full = pd.concat([
    pd.read_csv("../data/raw/bdshs/train.csv"),
    pd.read_csv("../data/raw/bdshs/val.csv"),
    pd.read_csv("../data/raw/bdshs/test.csv"),
], ignore_index=True)

bdshs_full["text_clean"] = bdshs_full["sentence"].apply(clean_text)

bidwesh = pd.read_csv("../data/raw/bidwesh/Regional Translated Texts.csv")
bidwesh["text_clean"] = bidwesh["Standard Bangla"].apply(clean_text)

bdshs_texts_set = set(bdshs_full["text_clean"])
bidwesh_texts_set = set(bidwesh["text_clean"])

# Find exactly overlapping texts directly
bidwesh_source_ids = bidwesh_texts_set.intersection(bdshs_texts_set)

# Save exclusion list
with open("../data/duplicate_exclusion_list.json", "w", encoding="utf-8") as f:
    json.dump(list(bidwesh_source_ids), f, ensure_ascii=False)
print(f"Exclusion list saved with {len(bidwesh_source_ids)} matched items.\n")

# Prepare and split task data
def prepare_task_data(task_name: str, raw_df: pd.DataFrame, bidwesh_exclusion_ids: set = None):
    out_dir = f"../data/processed/{task_name}"
    os.makedirs(out_dir, exist_ok=True)

    df = raw_df.copy()
    if "text_clean" not in df.columns:
        df["text_clean"] = df["text"].apply(clean_text)
        
    df = df.dropna(subset=["text_clean", "label"])
    df = df[df["text_clean"].str.len() > 0].reset_index(drop=True)
-
    before_dedup = len(df)
    df = df.drop_duplicates(subset=["text_clean"], keep="first").reset_index(drop=True)
    if before_dedup > len(df):
        print(f"[{task_name}] Dropped {before_dedup - len(df)} duplicate texts.")

    if task_name == "offensive" and bidwesh_exclusion_ids:
        before_len = len(df)
        df = df[~df["text_clean"].isin(bidwesh_exclusion_ids)].reset_index(drop=True)
        print(f"[{task_name}] Exclusion applied: {before_len} -> {len(df)} rows.")

    # Four-way split (Fixed proportions: 70/10/10/10)
    train_val_cal, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])
    train_val, cal_df = train_test_split(train_val_cal, test_size=0.111, random_state=42, stratify=train_val_cal["label"])
    train_df, val_df = train_test_split(train_val, test_size=0.125, random_state=42, stratify=train_val["label"])

    for split in [train_df, val_df, cal_df, test_df]:
        if "label" in split.columns:
            split.rename(columns={"label": "labels"}, inplace=True)

    for name, split in [("train", train_df), ("val", val_df), ("cal", cal_df), ("test", test_df)]:
        split.to_csv(f"{out_dir}/{name}.csv", index=False, encoding="utf-8")
        print(f"[{task_name}] {name}.csv saved: {len(split)} rows.")

    # E.4 verification gate
    ids = {name: set(split["text_clean"]) for name, split in [("train", train_df), ("val", val_df), ("cal", cal_df), ("test", test_df)]}
    pairs = [("train", "val"), ("train", "cal"), ("train", "test"), ("val", "cal"), ("val", "test"), ("cal", "test")]
    for a, b in pairs:
        assert len(ids[a] & ids[b]) == 0, f"Error: Overlap found between {a} and {b}!"
    print(f"[{task_name}] Split verification passed. No overlap found.\n")
    
    return train_df, val_df, cal_df, test_df

print("Preparing Sentiment Task Data (SentNoB)...")
sentnob_raw = load_dataset("khondoker/SentNoB")["train"].to_pandas().rename(columns={"Data": "text", "Label": "label"})
prepare_task_data("sentiment", sentnob_raw)

print("Preparing Offensive Task Data (BD-SHS)...")
bdshs_full = bdshs_full.rename(columns={"sentence": "text", "hate speech": "label"})
prepare_task_data("offensive", bdshs_full, bidwesh_source_ids)

Loading BD-SHS and BIDWESH for exclusion matching...
Cleaning texts... This should be fast.
Matching started...
Exclusion list saved with 3040 matched items.

Preparing Sentiment Task Data (SentNoB)...
[sentiment] Dropped 2128 duplicate texts.
[sentiment] train.csv saved: 9520 rows.
[sentiment] val.csv saved: 1361 rows.
[sentiment] cal.csv saved: 1359 rows.
[sentiment] test.csv saved: 1360 rows.
[sentiment] Split verification passed. No overlap found.

Preparing Offensive Task Data (BD-SHS)...
[offensive] Dropped 21 duplicate texts.
[offensive] Exclusion applied: 50260 -> 47220 rows.
[offensive] train.csv saved: 33057 rows.
[offensive] val.csv saved: 4723 rows.
[offensive] cal.csv saved: 4718 rows.
[offensive] test.csv saved: 4722 rows.
[offensive] Split verification passed. No overlap found.



(                                                    text  target  \
 30113    তোরে দেইখা মনে হয় বাংলাদেশে নায়কের অনেক অভাবআছে     NaN   
 36171                                              মুরুখ     NaN   
 19105  ফারহা খান নামটা পাল্টে কুত্তা নাম রাখিস আমাদের...  female   
 38741                        সাকিব না থাকলে বালো আমি আচি     NaN   
 22031                            আমি ছাই দুজন এক সাতে হক     NaN   
 ...                                                  ...     ...   
 17820  শালার পো কখন ফেসবুতে পোস্ট দিলি আর শেষে লিখলি ...    male   
 28974    ঢাকার ভক্তরা কমেন্ট করুন খুলনার ভক্তরা লাইক দিন     NaN   
 25581  এর পরেও কিছু তথাকথিত লোক কমেন্টে তাদের বংশপরিচ...     NaN   
 6958   এ হালার পুত তো অনেক কিছু শিক্ষা দিতেছে যত বড় ...    male   
 38090  সৎ পথে করিও ভ্রমন যদিও হয় দেরি অসৎ নারী করিওনা...     NaN   
 
                          type  labels  \
 30113                     NaN       0   
 36171                     NaN       0   
 19105        religion_slander       1   
 3

In [3]:
import os
import pandas as pd

os.makedirs("../data/for_translation", exist_ok=True)

d0_test = pd.read_csv("../data/processed/sentiment/test.csv")

sampled = d0_test.sample(n=760, random_state=42)

d1_df = pd.DataFrame({
    "source_d0_index": sampled.index,
    "source_d0_text": sampled["text_clean"],
    "label": sampled["labels"]
})

d1_df.to_csv("../data/for_translation/d1_sentences_to_translate.csv", index=False)

In [4]:
import random
import pandas as pd

random.seed(42)

BANGLA_KEYBOARD_NEIGHBORS = {
    "র": ["ড", "ত"],
    "ড": ["র", "ঠ"],
    "ত": ["র", "থ"],
    "থ": ["ত", "দ"],
    "দ": ["থ", "ধ"],
    "ধ": ["দ", "ন"],
    "ন": ["ধ", "ব"],
    "ব": ["ন", "ভ"],
    "ভ": ["ব", "ম"],
    "ম": ["ভ", "য"],
    "ক": ["খ"],
    "খ": ["ক", "গ"],
    "গ": ["খ", "ঘ"],
    "ঘ": ["গ", "ঙ"],
    "চ": ["ছ"],
    "ছ": ["চ", "জ"],
    "জ": ["ছ", "ঝ"],
    "ঝ": ["জ", "ঞ"],
    "ট": ["ঠ"],
    "ঠ": ["ট", "ড"],
    "প": ["ফ"],
    "ফ": ["প", "ব"],
    "স": ["শ", "ষ"],
    "শ": ["স", "ষ"],
    "অ": ["আ"],
    "আ": ["অ", "ই"],
    "ই": ["ঈ"],
    "উ": ["ঊ"],
}

def inject_char_noise(text: str, noise_level: float) -> str:
    chars = list(text)
    n_to_corrupt = max(1, int(len(chars) * noise_level))
    positions = random.sample(range(len(chars)), min(n_to_corrupt, len(chars)))
    
    for pos in sorted(positions, reverse=True):
        action = random.choice(["swap", "delete", "duplicate"])
        ch = chars[pos]
        if action == "swap" and ch in BANGLA_KEYBOARD_NEIGHBORS:
            chars[pos] = random.choice(BANGLA_KEYBOARD_NEIGHBORS[ch])
        elif action == "delete":
            del chars[pos]
        elif action == "duplicate":
            chars.insert(pos, ch)
            
    return "".join(chars)

NOISE_LEVELS = [0.05, 0.10, 0.20]

def generate_noise_sets(task_name: str):
    test_df = pd.read_csv(f"../data/processed/{task_name}/test.csv")
    for level in NOISE_LEVELS:
        pct = int(level * 100)
        noisy_df = test_df.copy()
        noisy_df["text_clean"] = noisy_df["text_clean"].apply(lambda t: inject_char_noise(t, level))
        noisy_df.to_csv(f"../data/processed/{task_name}/synthetic_noise_{pct:02d}pct.csv", index=False, encoding="utf-8")

generate_noise_sets("sentiment")
generate_noise_sets("offensive")

In [2]:
import re
import pandas as pd
from normalizer import normalize
from sklearn.model_selection import train_test_split

def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return normalize(text)

SENTIMENT_LABEL_MAP = {"Neutral": 0, "Positive": 1, "Negative": 2}
OFFENSIVE_LABEL_MAP = {"NOT": 0, "OFF": 1}

def prepare_shift_split(df: pd.DataFrame, text_col: str, label_col: str, out_prefix: str, label_map: dict = None):
    temp = df[[text_col, label_col]].dropna().copy()
    temp["text_clean"] = temp[text_col].apply(clean_text)
    temp = temp[temp["text_clean"].str.len() > 0]
    temp = temp.drop_duplicates(subset=["text_clean"], keep="first").reset_index(drop=True)
    
    if label_map is not None:
        temp["labels"] = temp[label_col].map(label_map)
    else:
        temp["labels"] = temp[label_col].astype(int)
        
    temp = temp.dropna(subset=["labels"]).reset_index(drop=True)
    temp["labels"] = temp["labels"].astype(int)

    cal_df, test_df = train_test_split(
        temp[["text_clean", "labels"]], 
        test_size=0.8, 
        random_state=42, 
        stratify=temp["labels"]
    )
    
    cal_df.to_csv(f"{out_prefix}_cal.csv", index=False, encoding="utf-8")
    test_df.to_csv(f"{out_prefix}_test.csv", index=False, encoding="utf-8")

sentmix_raw = pd.read_csv("../data/raw/sentmix3l/sentmix3l.csv")
prepare_shift_split(
    sentmix_raw, 
    text_col="text", 
    label_col="label", 
    out_prefix="../data/processed/sentiment/d2_codemixed", 
    label_map=SENTIMENT_LABEL_MAP
)

offmix_raw = pd.read_csv("../data/raw/offmix3l/offmix3l.csv")
prepare_shift_split(
    offmix_raw, 
    text_col="text", 
    label_col="label", 
    out_prefix="../data/processed/offensive/d2_codemixed", 
    label_map=OFFENSIVE_LABEL_MAP
)

bidwesh_raw = pd.read_csv("../data/raw/bidwesh/BIDWESH Dataset.csv")
for dialect in ["Chittagong", "Noakhali", "Barishal"]:
    prepare_shift_split(
        bidwesh_raw, 
        text_col=dialect, 
        label_col="hate speech", 
        out_prefix=f"../data/processed/offensive/d1_{dialect.lower()}"
    )